# B-PEFT Demo — Bayesian Parameter-Efficient Fine-Tuning
**Pre-Defence Demo | IUT | Mainul**

Run cells top-to-bottom. GPU runtime recommended (Colab T4 is fine).

---
## 0. Setup — Clone / Pull Repo & Install

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/beft-thesis.git"  # <-- fill in before using
REPO_DIR = "beft-thesis"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    os.chdir(REPO_DIR)
    os.system("git pull")
else:
    print("Cloning repo...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

---
## Task 1 — Config

In [ ]:
from src.config import CFG
print("Seed:", CFG.seed)
print("N-way:", CFG.num_classes, "  K-shot:", CFG.shots)
print("Feature dim:", CFG.feature_dim, "  Adapter rank:", CFG.adapter_rank)

---
## Task 2 — Utilities

In [ ]:
from src.utils import set_seed, get_device
set_seed(42)
device = get_device()
print("Device:", device)

---
## Task 3 — Data (CIFAR-FS episode + SVHN OOD)

Downloads ~170 MB CIFAR-100 and ~60 MB SVHN on first run.

In [ ]:
from src.data import get_cifar100_test, sample_episode, get_svhn_ood
from src.config import CFG

dataset = get_cifar100_test(CFG.data_root, CFG.image_size)
support_x, support_y, query_x, query_y = sample_episode(
    dataset, CFG.test_class_ids, CFG.num_classes,
    CFG.shots, CFG.query_per_class, CFG.seed
)
print("support_x:", support_x.shape)  # (25, 3, 224, 224)
print("query_x:  ", query_x.shape)    # (75, 3, 224, 224)

svhn_x = get_svhn_ood(CFG.data_root, CFG.image_size, CFG.ood_num_samples, CFG.seed)
print("svhn_x:   ", svhn_x.shape)     # (500, 3, 224, 224)

---
## Task 4 — Frozen ResNet18 Backbone

In [ ]:
import torch
from src.backbone import build_frozen_resnet18

backbone = build_frozen_resnet18()
dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = backbone(dummy)
assert out.shape == (2, 512), f"Bad shape: {out.shape}"
print("Backbone output shape:", out.shape, "  OK")

---
## Task 5 — Bottleneck Adapter

In [ ]:
import torch
from src.adapter import BottleneckAdapter

adapter = BottleneckAdapter(512, 16)
x = torch.randn(4, 512)
out = adapter(x)
assert out.shape == (4, 512)
assert torch.allclose(out, x, atol=1e-6), "Adapter not identity at init!"
n_params = sum(p.numel() for p in adapter.parameters())
print(f"Adapter params: {n_params:,}  (expect ~16,912)")

---
## Task 6 — Full Model (Backbone + Adapter + Head)

In [ ]:
import torch
from src.model import BPEFTModel
from src.utils import count_trainable_params

model = BPEFTModel(num_classes=5, mode="evidential")
dummy = torch.randn(8, 3, 224, 224)
out = model(dummy)
assert out.shape == (8, 5)
assert (out >= 0).all(), "Evidential output must be non-negative!"
n = count_trainable_params(model)
print(f"Output shape: {out.shape}  |  Trainable params: {n:,}  (should be < 20K)")

---
## Task 7 — Losses (Evidential MSE + KL + Softmax CE)

In [ ]:
import torch
from src.losses import evidential_mse_loss, softmax_ce_loss

evidence = torch.rand(4, 5) * 2
target_oh = torch.eye(5)[[0, 1, 2, 3]]
ev_loss = evidential_mse_loss(evidence, target_oh, num_classes=5, kl_weight=1.0)
print("Evidential loss:", ev_loss.item())
assert ev_loss.isfinite()

logits = torch.randn(4, 5)
targets = torch.tensor([0, 1, 2, 3])
ce_loss = softmax_ce_loss(logits, targets)
print("CE loss:        ", ce_loss.item())
assert ce_loss.isfinite()

---
## Task 9 — Training

Trains for 200 steps on the 5-way 5-shot support set.
Saves checkpoint to `checkpoints/model_{mode}.pt`.

In [ ]:
# Train evidential model
!python -m src.train --mode evidential

In [ ]:
# Train softmax baseline
!python -m src.train --mode softmax

---
## Task 8 — Metrics (sanity check on synthetic data)

In [ ]:
import torch
from src.metrics import accuracy, expected_calibration_error, brier_score, ood_auroc
import numpy as np

# Synthetic well-calibrated probs
probs = torch.softmax(torch.randn(50, 5), dim=-1)
targets = torch.randint(0, 5, (50,))
print(f"Accuracy:  {accuracy(probs, targets):.3f}")
print(f"ECE:       {expected_calibration_error(probs, targets):.3f}")
print(f"Brier:     {brier_score(probs, targets, 5):.3f}")

id_scores  = np.random.uniform(0.7, 1.0, 75)
ood_scores = np.random.uniform(0.0, 0.4, 500)
print(f"OOD AUROC: {ood_auroc(id_scores, ood_scores):.3f}  (expect > 0.9)")

---
## Task 10 — Evaluation & Plots

In [ ]:
!python -m src.evaluate

In [ ]:
import json
from IPython.display import Image, display

with open("results/metrics.json") as f:
    metrics = json.load(f)

print("=== RESULTS ===")
for mode in ["evidential", "softmax"]:
    m = metrics[mode]
    print(f"\n{mode.upper()}")
    print(f"  Accuracy : {m['accuracy']:.3f}")
    print(f"  ECE      : {m['ece']:.3f}")
    print(f"  Brier    : {m['brier']:.3f}")
    print(f"  OOD AUROC: {m['ood_auroc']:.3f}")

print("\nTrainable params:", metrics.get("trainable_params", {}))

In [ ]:
from IPython.display import Image, display
display(Image("results/reliability_plot.png"))
display(Image("results/ood_histogram.png"))
display(Image("results/training_curve.png"))